# Transcript Studio (English) — run + test on Google Colab

Unzips the project, installs it, starts the backend (which also serves the web page) and opens a public
link to it through `ngrok` (Colab doesn't let your browser reach `localhost`, so the tunnel is required).

**English audio only.** Other languages get a clear "please provide English audio" screen.

**Run the cells top to bottom.**

## 1. Upload the project zip
Run this cell, click **Choose Files**, and pick the project `.zip` from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 2. Unzip it

In [ ]:
import zipfile, os

zip_name = [f for f in uploaded.keys() if f.endswith(".zip")][0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(".")

os.chdir("audio-text-studio/backend")
print("Now in:", os.getcwd())

## 3. Install dependencies
`pyngrok` is only for this Colab tunnel (not needed on a normal deployment), so it's installed separately.

In [ ]:
!apt-get -qq install -y tesseract-ocr ffmpeg   # tesseract = read text from images/scanned PDFs (Text -> Audio)
!pip install -q -r requirements.txt
!pip install -q pyngrok

## 4. Choose how it transcribes

**Option A — free Groq key (recommended).** Fast, needs no GPU, and turns **AI insights ON**.
Get a key in a minute at https://console.groq.com (no card needed), then paste it when the cell asks.
Free-tier limits apply: audio files up to 25 MB, rate limits, no uptime guarantee.

**Option B — leave it blank for fully local mode.** No account at all, but slower, and **no AI insights**
(there is no model to write them). It installs `ffmpeg` + `faster-whisper`, uses the `large-v3` model if this
session has a GPU (turn one on first: *Runtime → Change runtime type → T4 GPU*) or `small` on CPU, and fixes the
`libcublas.so.12` GPU error some Colab sessions have (if that fix can't work it falls back to the CPU automatically).

The key is not shown on screen and is not saved in the notebook.

In [ ]:
from getpass import getpass
import os, subprocess

groq_key = getpass("Paste your Groq API key (or just press Enter for local mode): ").strip()

if groq_key:
    env_contents = f"""TRANSCRIPTION_BACKEND=openai
ANALYSIS_BACKEND=openai
OPENAI_BASE_URL=https://api.groq.com/openai/v1
OPENAI_API_KEY={groq_key}
TRANSCRIBE_MODEL=whisper-large-v3
TEXT_MODEL=openai/gpt-oss-120b
MAX_UPLOAD_MB=25
STORAGE_DIR=./storage
ENABLE_DIARIZATION=false
ALLOWED_ORIGINS=*
ACCESS_KEY=
RATE_LIMIT_PER_MINUTE=10
"""
    mode_summary = "Groq (free, hosted): whisper-large-v3 + openai/gpt-oss-120b. AI insights are ON."
else:
    # ---- fully local ----
    subprocess.run("apt-get -qq install -y ffmpeg", shell=True)
    # "av<19": PyAV 19 broke faster-whisper's audio decoding ("open() got an unexpected keyword
    # argument 'metadata_errors'"); pinning is the documented workaround.
    subprocess.run(["pip", "install", "-q", "faster-whisper", "av<19"])

    has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    model_size = "large-v3" if has_gpu else "small"
    print(f"GPU detected: {has_gpu}")

    if has_gpu:
        # Colab's GPU image doesn't always have the CUDA 12 libraries faster-whisper needs
        # ("Library libcublas.so.12 is not found"). Install them and point the server at them.
        subprocess.run(["pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"])
        probe = ("import os, nvidia.cublas.lib, nvidia.cudnn.lib; "
                 "print(os.path.dirname(nvidia.cublas.lib.__file__) + ':' + os.path.dirname(nvidia.cudnn.lib.__file__))")
        libs = subprocess.run(["python", "-c", probe], capture_output=True, text=True).stdout.strip()
        if libs:
            os.environ["LD_LIBRARY_PATH"] = libs + ":" + os.environ.get("LD_LIBRARY_PATH", "")
            print("GPU libraries ready.")
        else:
            print("Could not set up GPU libraries - transcription will fall back to CPU (slower, still works).")

    env_contents = f"""TRANSCRIPTION_BACKEND=local
ANALYSIS_BACKEND=local
LOCAL_WHISPER_MODEL={model_size}
MAX_UPLOAD_MB=100
STORAGE_DIR=./storage
ENABLE_DIARIZATION=false
ALLOWED_ORIGINS=*
ACCESS_KEY=
RATE_LIMIT_PER_MINUTE=10
"""
    mode_summary = (f"Local: {model_size} on " + ("GPU" if has_gpu else "CPU") +
                    ". No AI insights in this mode (paste a free Groq key above to get them).")

with open(".env", "w") as f:
    f.write(env_contents)
print("Wrote .env ->", mode_summary)

### (Recommended) Run the built-in self-checks
About a minute. Every line should say `PASS` and the last line should say **All backend checks passed.**
If a line says `FAIL`, copy that block of output and send it over.

In [ ]:
!python test_backend.py

## 5. (Optional) ngrok authtoken
ngrok normally needs a free account: sign up at https://dashboard.ngrok.com/signup, copy your authtoken from
https://dashboard.ngrok.com/get-started/your-authtoken, and paste it below. Leave it blank to try without.

In [ ]:
from getpass import getpass
from pyngrok import ngrok

ngrok_token = getpass("Paste your ngrok authtoken (or leave blank to skip): ")
if ngrok_token.strip():
    ngrok.set_auth_token(ngrok_token.strip())

## 6. Start the backend in the background
Starts `uvicorn` on port 8000 and gives it a few seconds to come up.

**If you change anything above after starting the server, run section 9 first** (it stops the old server and tunnel), then re-run sections 6 and 7.

In [ ]:
import subprocess, time, urllib.request

log_file = open("server.log", "w")
server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
time.sleep(6)
try:
    print("Health check:", urllib.request.urlopen("http://localhost:8000/api/health").read().decode())
except Exception as exc:
    print("Server isn't answering yet:", exc, "- run the logs cell in section 8 to see why.")
print("Server started (pid:", server.pid, ") - the next cell opens the public URL.")

## 7. Open a public tunnel
This is the link you open in your browser. Anyone with the link can use your app (and use up your free Groq
limits) while it's open — close the tunnel (section 9) when you're done testing.

In [ ]:
public_url = ngrok.connect(8000, "http")
print("Open this URL in your browser:")
print(public_url)

## 8. Use it
Open the printed URL. Upload or record a short **English** clip and run it. To see the other screen, upload a
clip in another language: you should get **"English audio only"** with a **Back to the start** button.
If anything misbehaves, run the cell below to see the backend's log.

In [ ]:
print(open("server.log").read()[-4000:] or "(log is empty)")

## 9. Shut everything down when you're done

In [ ]:
ngrok.kill()
server.terminate()
print("Tunnel closed, server stopped.")